<a href="https://colab.research.google.com/github/Maxipuce/TP1EX1_SYST_TR/blob/main/IA_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Création et importation des libraires

In [ ]:
!pip install tsaug
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
from tsaug import TimeWarp
import tensorflow.keras.backend as K
import numpy as np
import osCréation et importation des libraires


[ ]
!pip install tsaug
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts

Chargement et organisation des données


[ ]
def load_data():
    normal = pd.read_csv("/content/sample_data/ptbdb_normal.csv", header=None)
    abnormal = pd.read_csv("/content/sample_data/ptbdb_abnormal.csv", header=None)

    mitbih_train = pd.read_csv("/content/sample_data/mitbih_train.csv", header=None)
    mitbih_test = pd.read_csv("/content/sample_data/mitbih_test.csv", header=None)

    return normal, abnormal, mitbih_train, mitbih_test

normal, abnormal, mitbih_train, mitbih_test = load_data()
Séparation des données (validation, test...)


[ ]


Chargement et organisation des données

In [ ]:
def load_data():
    normal = pd.read_csv("/content/sample_data/ptbdb_normal.csv", header=None)
    abnormal = pd.read_csv("/content/sample_data/ptbdb_abnormal.csv", header=None)

    mitbih_train = pd.read_csv("/content/sample_data/mitbih_train.csv", header=None)
    mitbih_test = pd.read_csv("/content/sample_data/mitbih_test.csv", header=None)

    return normal, abnormal, mitbih_train, mitbih_test

normal, abnormal, mitbih_train, mitbih_test = load_data()

Séparation des données (validation, test...)

In [ ]:
def load_and_balance_data(normal_path, abnormal_path, train_size=0.8, val_test_split=0.5, random_state=42):
    """
    Load and balance ECG data with train/val/test splits (80/10/10)

    Args:
        normal_path: Path to normal ECG data
        abnormal_path: Path to abnormal ECG data
        train_size: Ratio of data for training (default 0.8)
        val_test_split: Ratio of remaining data to allocate to validation (default 0.5)
        random_state: Random seed for reproducibility

    Returns:
        X_train, X_val, X_test, y_train, y_val, y_test, class_weights
    """
    normal = pd.read_csv(normal_path, header=None)
    abnormal = pd.read_csv(abnormal_path, header=None)

    normal['label'] = 0
    abnormal['label'] = 1

    data = pd.concat([normal, abnormal], axis=0)
    data = data.sample(frac=1, random_state=random_state).reset_index(drop=True)

    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values

    X = X.reshape(X.shape[0], X.shape[1], 1)

    class_counts = np.bincount(y)
    total_samples = len(y)
    class_weights = {
        0: total_samples / (2 * class_counts[0]),  # Normal
        1: total_samples / (2 * class_counts[1])   # Abnormal
    }

    # First split: train vs (val + test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        train_size=train_size,
        random_state=random_state,
        stratify=y
    )

    # Second split: val vs test from remaining data
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=val_test_split,
        random_state=random_state,
        stratify=y_temp
    )

    print(f"Total samples: {len(X)}")
    print(f"Train samples: {len(X_train)} ({len(X_train)/len(X):.1%})")
    print(f"Validation samples: {len(X_val)} ({len(X_val)/len(X):.1%})")
    print(f"Test samples: {len(X_test)} ({len(X_test)/len(X):.1%})")

    return X_train, X_val, X_test, y_train, y_val, y_test, class_weights

X_train, X_val, X_test, y_train, y_val, y_test, class_weights = load_and_balance_data(
    "/content/sample_data/ptbdb_normal.csv",
    "/content/sample_data/ptbdb_abnormal.csv"
)

Ajout du Time Warping et Jittering
(amélioration de la robustesse et de la sensibilité aux perturbations)

In [ ]:
# Ajouter du bruit léger (jittering)
def add_jitter(X, noise_level=0.01):
    noise = np.random.normal(loc=0.0, scale=noise_level, size=X.shape)
    return X + noise

# Appliquer un time warping
def augment_with_time_warping(X, y):
    augmenter = TimeWarp(n_speed_change=3, max_speed_ratio=2)
    X_aug = augmenter.augment(X)
    y_aug = y.copy()
    return np.concatenate((X, X_aug), axis=0), np.concatenate((y, y_aug), axis=0)

# Appliquer augmentation sur le training set
X_train = add_jitter(X_train)
X_train, y_train = augment_with_time_warping(X_train, y_train)

Focal Loss (Réduit l'impact des exemples bien classés et augmente celui des exemples mal classés)

In [ ]:
def focal_loss(gamma=2., alpha=.25):
    def focal_loss_fixed(y_true, y_pred):
        y_true = K.cast(y_true, dtype='float32')
        epsilon = K.epsilon()
        y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
        pt = tf.where(K.equal(y_true, 1), y_pred, 1 - y_pred)
        return -K.mean(alpha * K.pow(1. - pt, gamma) * K.log(pt))
    return focal_loss_fixed

Model Architecture

In [ ]:
from tensorflow.keras import regularizers

In [ ]:
def build_model(input_shape):
    model = models.Sequential([
        # First Conv Block
        layers.Conv1D(64, kernel_size=15, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.4),

        # Second Conv Block
        layers.Conv1D(128, kernel_size=11, activation='relu',kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.4),
                # Third Conv Block
        layers.Conv1D(256, kernel_size=7, activation='relu',kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.4),

        layers.GlobalAveragePooling1D(),

        layers.Dense(128, activation='relu',kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(1, activation='sigmoid')
    ])

    # Définir le scheduler
    lr_schedule = CosineDecayRestarts(
        initial_learning_rate=0.001,
        first_decay_steps=10,
        t_mul=2.0,
        m_mul=1.0,
        alpha=1e-6
    )

    # Créer l’optimiseur AVEC le scheduler
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    # Compiler le modèle AVEC focal loss ou autre
    model.compile(
        optimizer=optimizer,
        loss=focal_loss(gamma=2.0, alpha=0.25),  # ou 'binary_crossentropy'
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [ ]:
input_shape = (X_train.shape[1], X_train.shape[2])
model = build_model(input_shape)
model.summary()

Exécution de l'entraînement


In [ ]:
"Stopper l'entraînement intelligemment"
early_stopping = callbacks.EarlyStopping(
    monitor='val_auc',
    patience=15,
    mode='max',
    restore_best_weights=True)

"Sauvegarder automatiquement le meilleur modèle"
model_checkpoint = callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_auc',
    save_best_only=True,
    mode='max')


"Ajuster dynamiquement le learning rate"
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping, model_checkpoint, reduce_lr],shuffle=True)

TRAINING HISTORY

In [ ]:
def plot_history(history):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_history(history)

CONFUSION MATRIX

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(model, X_test, y_test):
    # Prédictions
    y_pred = model.predict(X_test)
    y_pred_classes = (y_pred > 0.5).astype(int)

    # Matrice de confusion
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Abnormal'],
                yticklabels=['Normal', 'Abnormal'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

plot_confusion_matrix(model, X_test, y_test)